<a href="https://colab.research.google.com/github/Hadi6618/PRISM/blob/main/ShanghaiTech_Ensemble_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STG-NF + MULDE Frame-Level Ensemble Fusion (PRISM)

This notebook fuses frame-level anomaly scores from **STG-NF** (pose level)
and **MULDE** (appearance level) for either the **ShanghaiTech Campus** or
**Avenue** dataset.

**Pipeline:**
1. Load the two score pickles produced by the upstream training/export notebooks.
2. Align the two streams per `(video_id, frame_index)`. Video-ID conventions
   are auto-detected and remapped (Avenue: STG-NF `01_0001` <-> MULDE `01`).
3. Apply **global rank** normalization so both streams live on `[0, 1]`.
4. Grid-search an **independent** Gaussian smoothing sigma per model
   (`sigma_stgnf`, `sigma_mulde`). The two streams have different noise
   profiles — STG-NF scores are pre-smoothed by their eval pipeline while
   MULDE scores are raw — so a single shared sigma is a compromise.
5. Grid-search the fusion weight `beta_1` over `[0, 1]` in 0.01 steps.
6. Save the grid table (`fusion_grid_search.csv`) and report
   (`fusion_report.json`) to Google Drive.

**Set `DATASET` in the config cell to choose the dataset.** All results are
saved under `/content/drive/MyDrive/Fusion/runs/<DATASET>/ensemble/`.

In [1]:
import subprocess
import importlib.util
import sys
import warnings
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/Hadi6618/PRISM.git'
REPO_DIR = Path('/content/PRISM')


Mounted at /content/drive


In [2]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR} already exists, skipping clone.')


In [3]:
_FUSION_PATH = REPO_DIR / 'fusion.py'
if not _FUSION_PATH.exists():
    raise FileNotFoundError(f'fusion.py not found at {_FUSION_PATH}')
spec = importlib.util.spec_from_file_location('fusion', _FUSION_PATH)
fusion = importlib.util.module_from_spec(spec)
sys.modules['fusion'] = fusion
spec.loader.exec_module(fusion)
print(f'Loaded fusion module from {_FUSION_PATH}')


Loaded fusion module from /content/PRISM/fusion.py


## Dataset selection

Set `DATASET` to `'ShanghaiTech'` or `'Avenue'`. The paths for each dataset are
pre-configured below — edit them only if your Drive layout differs.

In [14]:
# ======================================================================
# DATASET SELECTOR -- change this one line to switch datasets
# ======================================================================
DATASET = 'ShanghaiTech'   # 'ShanghaiTech' or 'Avenue'

# Per-dataset paths on Google Drive. Edit if your layout differs.
DATASET_PATHS = {
    'ShanghaiTech': {
        'stgnf_pkl': Path('/content/drive/MyDrive/STG-NF/original_shanghaitech/logs/shanghaitech_stgnf_scores_84.pkl'),
        'mulde_pkl':  Path('/content/drive/MyDrive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/shanghaitech_mulde_scores_79_7.pkl'),
        'output_dir': Path('/content/drive/MyDrive/Fusion/runs/ShanghaiTech/ensemble'),
    },
    'Avenue': {
        'stgnf_pkl': Path('/content/drive/MyDrive/STG-NF/Avenue_dataset/logs/avenue_stgnf_scores_57.pkl'),
        'mulde_pkl':  Path('/content/drive/MyDrive/MULDE/runs/avenue_hiera_l_mulde/Final_avenue_scores/artifacts/avenue_mulde_scores_81_4.pkl'),
        'output_dir': Path('/content/drive/MyDrive/Fusion/runs/Avenue/ensemble'),
    },
}

assert DATASET in DATASET_PATHS, f'DATASET must be one of {list(DATASET_PATHS.keys())}'
cfg = DATASET_PATHS[DATASET]
STGNF_PKL = cfg['stgnf_pkl']
MULDE_PKL = cfg['mulde_pkl']
OUTPUT_DIR = cfg['output_dir']

for p in [STGNF_PKL, MULDE_PKL]:
    if not p.exists():
        raise FileNotFoundError(f'Missing score file: {p}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset:      {DATASET}')
print(f'STG-NF scores: {STGNF_PKL}')
print(f'MULDE scores:  {MULDE_PKL}')
print(f'Output dir:    {OUTPUT_DIR}')


Dataset:      ShanghaiTech
STG-NF scores: /content/drive/MyDrive/STG-NF/original_shanghaitech/logs/shanghaitech_stgnf_scores_84.pkl
MULDE scores:  /content/drive/MyDrive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/shanghaitech_mulde_scores_79_7.pkl
Output dir:    /content/drive/MyDrive/Fusion/runs/ShanghaiTech/ensemble


In [15]:
stgnf_scores, stgnf_meta = fusion.load_score_pickle(STGNF_PKL)
mulde_scores, mulde_meta = fusion.load_score_pickle(MULDE_PKL)
print(f'STG-NF videos: {len(stgnf_scores)}  (sample IDs: {list(stgnf_scores.keys())[:3]})')
print(f'MULDE  videos: {len(mulde_scores)}  (sample IDs: {list(mulde_scores.keys())[:3]})')


STG-NF videos: 107  (sample IDs: ['01_0014', '01_0015', '01_0016'])
MULDE  videos: 107  (sample IDs: ['01_0014', '01_0015', '01_0016'])


In [16]:
# Align per (video_id, frame_index). auto_detect_offset sweeps offset in {-2..+2}
# and tries both anomaly/normality polarity for STG-NF, picking the combination
# that maximises STG-NF's standalone Micro AUC. Video-ID aliases are applied
# automatically (Avenue: 01_0001 <-> 01).
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    aligned, align_stats = fusion.align_per_video(
        stgnf_scores,
        mulde_scores,
        auto_detect_offset=True,
        stgnf_score_mode='auto',
    )

n_remap = align_stats.get('video_id_mapping_applied', 0)
print(
    f"Aligned videos: {align_stats['videos_aligned']} "
    f"(STG-NF={align_stats['videos_in_stgnf']}, MULDE={align_stats['videos_in_mulde']})"
)
print(f"  stgnf_frame_offset = {align_stats.get('stgnf_frame_offset', 0)}")
print(f"  stgnf_score_mode   = {align_stats.get('stgnf_score_mode', 'auto')}")
print(f"  video_ids_remapped = {n_remap}")
print(f"  label_inversion    = {align_stats.get('label_inversion_detected', False)}")
if 'auto_detect' in align_stats:
    print('  Offset/polarity candidates:')
    for key, payload in align_stats['auto_detect']['stgnf_micro_auc_per_candidate'].items():
        off, mode = key.split('|', 1)
        auc = payload.get('micro_auc_stgnf')
        auc_s = f'{auc * 100:.4f}%' if auc is not None else 'n/a'
        marker = ' <-- chosen' if (
            int(off) == align_stats.get('stgnf_frame_offset', 0)
            and mode == align_stats.get('stgnf_score_mode', 'auto')
        ) else ''
        print(f'    offset={int(off):+d}  mode={mode:9s}  AUC={auc_s}{marker}')


Aligned videos: 107 (STG-NF=107, MULDE=107)
  stgnf_frame_offset = 0
  stgnf_score_mode   = normality
  video_ids_remapped = 0
  label_inversion    = True
  Offset/polarity candidates:
    offset=-2  mode=anomaly    AUC=16.3200%
    offset=-2  mode=normality  AUC=83.6800%
    offset=-1  mode=anomaly    AUC=16.1813%
    offset=-1  mode=normality  AUC=83.8187%
    offset=+0  mode=anomaly    AUC=16.0739%
    offset=+0  mode=normality  AUC=83.9261% <-- chosen
    offset=+1  mode=anomaly    AUC=16.0304%
    offset=+1  mode=normality  AUC=83.9696%
    offset=+2  mode=anomaly    AUC=16.0184%
    offset=+2  mode=normality  AUC=83.9816%


In [17]:
# Global rank normalization: converts each model's raw scores to [0,1] percentiles.
# Must happen BEFORE Gaussian smoothing so outlier spikes are capped at 1.0.
NORMALIZATION = 'global_rank'
aligned = fusion.apply_normalization(aligned, strategy=NORMALIZATION)
print(f'Normalization strategy: {NORMALIZATION}')

_y = np.concatenate([v.labels for v in aligned])
_s = np.concatenate([v.stgnf_scores for v in aligned])
_m = np.concatenate([v.mulde_scores for v in aligned])
if len(np.unique(_y)) >= 2:
    print(f'STG-NF alone (after norm) Micro AUC: {roc_auc_score(_y, _s) * 100:.4f}%')
    print(f'MULDE  alone (after norm) Micro AUC: {roc_auc_score(_y, _m) * 100:.4f}%')


Normalization strategy: global_rank
STG-NF alone (after norm) Micro AUC: 83.9261%
MULDE  alone (after norm) Micro AUC: 78.9810%


In [18]:
# Grid-search an INDEPENDENT (sigma_stgnf, sigma_mulde) pair.
# The two streams have different noise profiles — STG-NF scores are
# pre-smoothed by their eval pipeline while MULDE scores are raw — so
# sharing one sigma would over-smooth STG-NF or under-smooth MULDE.
print('Searching for best Gaussian smoothing (sigma_stgnf, sigma_mulde) ...')
best_sigma, sigma_results = fusion.search_best_sigma_independent(
    aligned,
    sigma_candidates=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15),
    normalization=None,  # already normalized above
)

sigma_stgnf, sigma_mulde = best_sigma
print(f'Best independent sigma = {best_sigma} (sigma_stgnf={sigma_stgnf}, sigma_mulde={sigma_mulde})')

aligned = fusion.smooth_scores_independent(aligned, sigma_stgnf, sigma_mulde)


Searching for best Gaussian smoothing sigma ...
  Independent sigma search: 16x16 = 256 combinations
  σ_stgnf  σ_mulde      STG-NF       MULDE         avg
     12.0     14.0    84.2703%    80.6891%    82.4797% <--
     12.0     13.0    84.2703%    80.6858%    82.4780%
     13.0     14.0    84.2667%    80.6891%    82.4779%
     12.0     15.0    84.2703%    80.6840%    82.4771%
     11.0     14.0    84.2640%    80.6891%    82.4765%
     13.0     13.0    84.2667%    80.6858%    82.4762%
     13.0     15.0    84.2667%    80.6840%    82.4753%
     11.0     13.0    84.2640%    80.6858%    82.4749%
     11.0     15.0    84.2640%    80.6840%    82.4740%
     14.0     14.0    84.2544%    80.6891%    82.4717%
     12.0     12.0    84.2703%    80.6722%    82.4713%
     14.0     13.0    84.2544%    80.6858%    82.4701%
     10.0     14.0    84.2503%    80.6891%    82.4697%
     13.0     12.0    84.2667%    80.6722%    82.4695%
     14.0     15.0    84.2544%    80.6840%    82.4692%
Best sigma = (1

In [19]:
beta_1_values = list(np.round(np.arange(0.0, 1.0 + 1e-9, 0.01), 4))
results, best, summary = fusion.grid_search_fusion(aligned, beta_1_values=beta_1_values)
summary['dataset'] = DATASET
summary['normalization'] = NORMALIZATION
summary['smooth_sigma'] = best_sigma
fusion.write_outputs(
    results=results,
    best=best,
    summary=summary,
    alignment_stats=align_stats,
    stgnf_meta=stgnf_meta,
    mulde_meta=mulde_meta,
    output_dir=OUTPUT_DIR,
)


Saved grid search table: /content/drive/MyDrive/Fusion/runs/ShanghaiTech/ensemble/fusion_grid_search.csv
Saved ensemble report:   /content/drive/MyDrive/Fusion/runs/ShanghaiTech/ensemble/fusion_report.json
Optimal weights -> beta_1 (STG-NF)=0.54, beta_2 (MULDE)=0.46, Micro AUC=89.8320%


In [20]:
import pandas as pd
table = fusion.results_to_table(results)
table_sorted = table.dropna(subset=['micro_auc']).sort_values('micro_auc', ascending=False)
display(table_sorted.head(30))


,beta_1_stgnf,beta_2_mulde,micro_auc
54,0.54,0.46,0.898320
55,0.55,0.45,0.898318
53,0.53,0.47,0.898234
56,0.56,0.44,0.898205
52,0.52,0.48,0.898058
57,0.57,0.43,0.898000
51,0.51,0.49,0.897782
58,0.58,0.42,0.897715
50,0.50,0.50,0.897417
59,0.59,0.41,0.897343


In [13]:
y = np.concatenate([v.labels for v in aligned])
s = np.concatenate([v.stgnf_scores for v in aligned])
m = np.concatenate([v.mulde_scores for v in aligned])
print(f'Dataset:    {DATASET}')
print(f'Frames:     {y.shape[0]}')
print(f'Videos:     {len(aligned)}')
print(f'STG-NF AUC: {roc_auc_score(y, s) * 100:.4f}%')
print(f'MULDE  AUC: {roc_auc_score(y, m) * 100:.4f}%')
print(f'Sigma:      {best_sigma}')
if best is not None and best.micro_auc is not None:
    print(f'Ensemble:   {best.micro_auc * 100:.4f}% (beta_1={best.beta_1:.2f}, beta_2={best.beta_2:.2f})')
print(f'Correlation (STG-NF vs MULDE): {np.corrcoef(s, m)[0, 1]:.4f}')


Dataset:    Avenue
Frames:     15324
Videos:     21
STG-NF AUC: 56.9552%
MULDE  AUC: 82.5757%
Sigma:      (0.0, 11.0)
Ensemble:   82.7512% (beta_1=0.07, beta_2=0.93)
Correlation (STG-NF vs MULDE): 0.0883
